# FantasAI Weekly Projections Ingestion

Pull weekly fantasy projections from FantasAI Cloudflare Worker API (Sleeper data source) and land it in bronze and silver Delta tables.

In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

BASE_URL = "https://api.fantasai.net"

# Parameters for the upcoming week
WEEK = 1
SEASON = 2025
TOP_N = 100  # Number of top players to fetch

In [0]:
# Create bronze projections table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.bronze_projections (
  player_id STRING,
  player_name STRING,
  team STRING,
  position STRING,
  week INT,
  season INT,
  projected_points DOUBLE,
  projections STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

# Create silver projections table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.silver_projections (
  player_id STRING,
  player_name STRING,
  team STRING,
  position STRING,
  week INT,
  season INT,
  projected_points DOUBLE,
  projections STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

print("✓ Tables created")

In [0]:
# Fetch weekly projections from Cloudflare Worker API
import json

response = requests.get(
    f"{BASE_URL}/api/v1/projections",
    params={"week": WEEK, "season": SEASON, "top": TOP_N},
    timeout=30
)
response.raise_for_status()
payload = response.json()

print(f"Fetched projections for {len(payload)} players")

# Convert to DataFrame rows
rows = []
for proj in payload:
    rows.append(
        Row(
            player_id=str(proj.get("player_id") or proj.get("id", "")),
            player_name=proj.get("player_name") or proj.get("name"),
            team=proj.get("team"),
            position=proj.get("position"),
            week=WEEK,
            season=SEASON,
            projected_points=float(proj.get("projected_points", 0) or proj.get("points", 0)),
            projections=json.dumps(proj.get("projections", {})),  # Store detailed projections as JSON
        )
    )

projections_df = spark.createDataFrame(rows)

display(projections_df)

In [0]:
# Write to bronze table
bronze_df = projections_df.withColumn("ingested_at", F.current_timestamp())

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("main.fantasai.bronze_projections")
)

print(f"✓ Wrote {bronze_df.count()} records to bronze_projections")

In [0]:
# Transform and deduplicate for silver (latest projections for player/week/season)
silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("player_name").cast("string"),
        F.col("team").cast("string"),
        F.col("position").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("projected_points").cast("double"),
        F.col("projections").cast("string"),
        F.col("ingested_at"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

display(silver_df)

In [0]:
# Write to silver table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")  # Overwrite to keep only latest projections
    .saveAsTable("main.fantasai.silver_projections")
)

print(f"✓ Wrote {silver_df.count()} records to silver_projections")